In [1]:
!git clone https://github.com/VinAIResearch/COVID19Tweet.git
%cd COVID19Tweet
!unzip -q WNUT-2020-Task-2-Dataset.zip -d data
!find data -type f

Cloning into 'COVID19Tweet'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 47 (delta 14), reused 26 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 6.42 MiB | 16.44 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/COVID19Tweet
data/WNUT-2020-Task-2-Dataset/valid.tsv
data/WNUT-2020-Task-2-Dataset/unlabeled_test_with_noise.tsv
data/WNUT-2020-Task-2-Dataset/test.tsv
data/WNUT-2020-Task-2-Dataset/train.tsv


In [2]:
# 1. 함수 가져오기
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report #정확도, 점수, 성능표 보기

In [3]:
# 2. 데이터 읽기
train_df = pd.read_csv("data/WNUT-2020-Task-2-Dataset/train.tsv", sep="\t") #탭으로 나눠져있음
valid_df = pd.read_csv("data/WNUT-2020-Task-2-Dataset/valid.tsv", sep="\t", header=None)
valid_df.columns = ['Id', 'Text', 'Label'] #헤더가 없다고 함 + 열 이름 할당하게 되어 이름도 다시 할당

In [4]:
# 3. 데이터 확인
train_df.head()
train_df.columns
train_df.shape

(6936, 3)

In [5]:
# test 데이터 읽기
test_df = pd.read_csv("data/WNUT-2020-Task-2-Dataset/test.tsv", sep="\t", header=None)
test_df.columns = ["Id", "Text", "Label"]

In [6]:
#인코더 가져오기

!pip install -q transformers datasets accelerate emoji==0.6.0
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, classification_report
# 라벨 숫자 변환
label2id = {
    "UNINFORMATIVE": 0,
    "INFORMATIVE": 1
}

id2label = {
    0: "UNINFORMATIVE",
    1: "INFORMATIVE"
}

train_bert = train_df[["Text", "Label"]].dropna().copy()
valid_bert = valid_df[["Text", "Label"]].dropna().copy()
test_bert = test_df[["Text", "Label"]].dropna().copy()

train_bert["labels"] = train_bert["Label"].map(label2id)
valid_bert["labels"] = valid_bert["Label"].map(label2id)
test_bert["labels"] = test_bert["Label"].map(label2id)
# 코로나 트윗 특화 인코더
model_name = "vinai/bertweet-covid19-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

m1_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-covid19-base-cased
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstrea

In [7]:
#인코더는 프리즈
for param in m1_model.base_model.parameters():
    param.requires_grad = False

In [8]:
#토큰화 하기
train_dataset = Dataset.from_pandas(train_bert[["Text", "labels"]], preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_bert[["Text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_bert[["Text", "labels"]], preserve_index=False)
def tokenize(batch):
    return tokenizer(
        batch["Text"],
        truncation=True,
        max_length=128
    )

train_tokenized = train_dataset.map(tokenize, batched=True, remove_columns=["Text"])
valid_tokenized = valid_dataset.map(tokenize, batched=True, remove_columns=["Text"])
test_tokenized = test_dataset.map(tokenize, batched=True, remove_columns=["Text"])

Map:   0%|          | 0/6936 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [9]:
#학습 코드
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "informative_f1": f1_score(labels, preds, pos_label=1),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./m1_bertweet_covid_freeze",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=m1_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss
50,0.592739
100,0.500080
150,0.438749
200,0.472876
250,0.393230
300,0.467648
350,0.380850
400,0.396257
450,0.419472
500,0.385882


TrainOutput(global_step=1302, training_loss=0.39580041820186257, metrics={'train_runtime': 117.6836, 'train_samples_per_second': 176.813, 'train_steps_per_second': 11.064, 'total_flos': 793066054177920.0, 'train_loss': 0.39580041820186257, 'epoch': 3.0})

In [10]:
#테스트 결과
print("VALID 결과")
valid_result = trainer.evaluate(valid_tokenized)
print(valid_result)

print("TEST 결과")
test_result = trainer.evaluate(test_tokenized)
print(test_result)

VALID 결과


Training Loss,Validation Loss,Step,Accuracy,Informative F1,Macro F1
0.337137,0.389717,1302,0.829000,0.822060,0.828740


{'eval_loss': 0.38971662521362305, 'eval_accuracy': 0.829, 'eval_informative_f1': 0.8220603537981269, 'eval_macro_f1': 0.8287395127989672}
TEST 결과


Training Loss,Validation Loss,Step,Accuracy,Informative F1,Macro F1
0.337137,0.429662,1302,0.797500,0.787625,0.797061


{'eval_loss': 0.42966169118881226, 'eval_accuracy': 0.7975, 'eval_informative_f1': 0.7876245411641322, 'eval_macro_f1': 0.7970611955701215}
